In [1]:
import onnxruntime as ort

# Tải mô hình ONNX
model_path = "wbnet_model.onnx"
session = ort.InferenceSession(model_path)

# In thông tin về input và output
print("Inputs:", session.get_inputs())
print("Outputs:", session.get_outputs())


Inputs: [<onnxruntime.capi.onnxruntime_pybind11_state.NodeArg object at 0x000001D850F34F70>]
Outputs: [<onnxruntime.capi.onnxruntime_pybind11_state.NodeArg object at 0x000001D844D37970>, <onnxruntime.capi.onnxruntime_pybind11_state.NodeArg object at 0x000001D844D343F0>]


In [2]:
import numpy as np

# Dummy input (batch size = 1, channels = 9, height =256, width =256)
input_name = session.get_inputs()[0].name
dummy_input = np.random.rand(1, 9, 256, 256).astype(np.float32)

# Dự đoán với ONNX Runtime
output = session.run(None, {input_name: dummy_input})
print("Output Shape:", np.array(output).shape)


Output Shape: (2, 1, 3, 256, 256)


In [8]:
from torch import nn
import torch
from models.wb_net import WBNet

# Chạy mô hình PyTorch gốc
pytorch_model = WBNet()
pytorch_model.eval()
# WBNet trả về (out_img, weights)
torch_output, _ = pytorch_model(torch.from_numpy(dummy_input))


# So sánh output của ONNX và PyTorch
onnx_output = output[0]
print("Are outputs close?", np.allclose(onnx_output, torch_output.detach().numpy(), atol=1e-6))

# print(onnx_output.shape)


Are outputs close? False


In [4]:
import time

start = time.time()
for _ in range(100):  # Chạy 100 lần inference
    session.run(None, {input_name: dummy_input})
end = time.time()

print(f"Average inference time: {(end - start) / 100:.4f} seconds")


Average inference time: 0.0189 seconds


In [11]:
img = torch.randn(9, 320, 320)
img

tensor([[[ 1.3973e+00,  9.8763e-01, -4.6777e-01,  ..., -1.2930e+00,
           1.4181e+00,  5.9981e-01],
         [ 6.3043e-01, -4.3258e-02,  8.7300e-01,  ...,  3.1795e-01,
           5.7673e-01, -4.1762e-01],
         [ 2.8952e-01,  4.2426e-01, -4.6891e-02,  ..., -4.4009e-01,
          -8.4526e-01, -2.8825e-01],
         ...,
         [-6.0126e-01,  2.5193e-01,  1.1903e+00,  ...,  1.4364e+00,
           9.2390e-03, -1.9906e+00],
         [-1.4332e+00,  3.9462e-01, -5.7474e-01,  ..., -3.0014e-01,
           9.5703e-01,  1.7740e+00],
         [ 1.7370e-01,  3.3814e-01, -3.6004e-01,  ...,  7.0415e-01,
           1.5025e-01,  5.3102e-01]],

        [[ 2.7332e-01,  1.6676e-01,  4.9719e-01,  ...,  1.2246e+00,
          -4.1975e-01,  1.5688e+00],
         [ 5.6412e-01, -2.8469e+00, -1.2013e-01,  ...,  2.0923e-01,
           2.7519e-01, -6.7373e-01],
         [-3.0053e+00,  2.4412e-02,  4.4528e-01,  ..., -3.6809e-01,
          -9.6073e-01, -6.6254e-01],
         ...,
         [-6.2596e-01,  7

In [13]:
import onnxruntime
import numpy as np
import torch

# Tải mô hình ONNX
onnx_path = "wbnet_model.onnx"
session = onnxruntime.InferenceSession(onnx_path)

# Lấy thông tin về đầu vào và đầu ra
input_name = session.get_inputs()[0].name
output_name = session.get_outputs()[0].name

# Tạo dummy input (giống với input khi export)
dummy_input = np.random.rand(1, 9, 320, 320).astype(np.float32)

# Chạy inference
output = session.run([output_name], {input_name: dummy_input})

# Kiểm tra output
print("Output shape:", np.array(output).shape)
print("Inference successful!")


Output shape: (1, 1, 3, 320, 320)
Inference successful!


In [29]:
import onnxruntime
import numpy as np
import cv2
import torch

# Đường dẫn tới mô hình ONNX
onnx_path = "wbnet_model.onnx"

# Khởi tạo ONNX Runtime session
session = onnxruntime.InferenceSession(onnx_path)

# Lấy tên input và output của mô hình
input_name = session.get_inputs()[0].name
output_name = session.get_outputs()[0].name

def preprocess_image(image_paths, t_size=(320, 320)):
    """
    Pre-process các ảnh đầu vào
    Args:
        image_paths: Danh sách đường dẫn tới các ảnh
        target_size: Kích thước cần resize
    
    Returns:
        inp_model: Dữ liệu đầu vào cho mô hình ONNX, dạng numpy array
    """
# Đọc và resize từng ảnh
    # Đọc và resize từng ảnh
    imgs = [cv2.resize(cv2.imread(img_path), (t_size, t_size)) for img_path in image_paths]
    # Gộp các ảnh thành một tensor với thứ tự kênh phù hợp
    stacked_imgs = np.concatenate([img.transpose(2, 0, 1) for img in imgs], axis=0)  # Shape: (9, t_size, t_size)
    # Thêm chiều batch size
    return np.expand_dims(stacked_imgs, axis=0).astype(np.float32)  # Shape: (1, 9, t_size, t_size)

def postprocess_output(output):
    """
    Xử lý tensor đầu ra từ mô hình thành ảnh RGB.
    """
    # Loại bỏ batch dimension, còn (3, 320, 320)
    # output = np.squeeze(output, axis=0)  
    # Chuyển từ (3, 320, 320) -> (320, 320, 3)
    output = output.transpose(1, 2, 0)  
    # Chuyển đổi giá trị về khoảng [0, 255] và thành uint8
    output = (output * 255).clip(0, 255).astype(np.uint8)  
    return output


# Đường dẫn tới các ảnh đầu vào
image_paths = ["E:\\refactor\\refactor\\src\\datahub\\test_data\\1\\1_1.png",
                "E:\\refactor\\refactor\\src\\datahub\\test_data\\1\\1_1.png", 
                "E:\\refactor\\refactor\\src\\datahub\\test_data\\1\\1_1.png"]

# Pre-Processing
inp_model = preprocess_image(image_paths, t_size=320)

# Chạy inference
output = session.run([output_name], {input_name: inp_model})[0]
print("Output shape:", output.shape)

print("Original Output Shape:", output.shape)  # Trước squeeze
output = np.squeeze(output, axis=0)  
print("After Squeeze:", output.shape)  # Sau khi squeeze
output = output.transpose(1, 2, 0)  
print("After Transpose:", output.shape)  # Sau khi transpose

    

# Gọi hàm xử lý đầu ra
result_image = postprocess_output(output)

# Lưu ảnh với OpenCV (chuyển từ RGB sang BGR)
cv2.imwrite("result_image.jpg", cv2.cvtColor(result_image, cv2.COLOR_RGB2BGR))

print("Inference completed! Result saved to result_image.jpg")



Output shape: (1, 3, 320, 320)
Original Output Shape: (1, 3, 320, 320)
After Squeeze: (3, 320, 320)
After Transpose: (320, 320, 3)


error: OpenCV(4.10.0) d:\a\opencv-python\opencv-python\opencv\modules\imgproc\src\color.simd_helpers.hpp:92: error: (-2:Unspecified error) in function '__cdecl cv::impl::`anonymous-namespace'::CvtHelper<struct cv::impl::`anonymous namespace'::Set<3,4,-1>,struct cv::impl::A0x46dff480::Set<3,4,-1>,struct cv::impl::A0x46dff480::Set<0,2,5>,4>::CvtHelper(const class cv::_InputArray &,const class cv::_OutputArray &,int)'
> Invalid number of channels in input image:
>     'VScn::contains(scn)'
> where
>     'scn' is 320


In [1]:
import torch
import torch.nn.functional as F

# Tạo tensor giả lập một ảnh 2D (batch=1, channels=1, height=4, width=4)
input_tensor = torch.arange(16, dtype=torch.float32).view(1, 1, 4, 4)

# Thay đổi kích thước
output_tensor = F.interpolate(input_tensor, size=(8, 8), mode='bilinear', align_corners=False)

print("Input tensor shape:", input_tensor.shape)
print("Output tensor shape:", output_tensor.shape)


Input tensor shape: torch.Size([1, 1, 4, 4])
Output tensor shape: torch.Size([1, 1, 8, 8])


In [2]:
output_tensor = F.interpolate(input_tensor, scale_factor=2, mode='bilinear', align_corners=False)
print("Output tensor shape with scale_factor:", output_tensor.shape)


Output tensor shape with scale_factor: torch.Size([1, 1, 8, 8])


In [9]:
import numpy as np
gamma_values = [2.5, 5.5]
for idx, gamma in enumerate(gamma_values, start=1):
    # print(gamma)
    gamma_table = np.array([((i / 255.0) ** gamma) * 255 for i in range(256)], dtype=np.uint8)
    print(gamma_table)

[  0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0
   0   0   0   0   0   0   0   0   0   0   1   1   1   1   1   1   1   1
   1   2   2   2   2   2   2   2   3   3   3   3   3   4   4   4   4   5
   5   5   5   6   6   6   6   7   7   7   8   8   8   9   9   9  10  10
  10  11  11  11  12  12  13  13  14  14  14  15  15  16  16  17  17  18
  18  19  19  20  21  21  22  22  23  23  24  25  25  26  27  27  28  29
  29  30  31  31  32  33  34  34  35  36  37  37  38  39  40  41  42  42
  43  44  45  46  47  48  49  50  51  52  52  53  54  55  56  57  59  60
  61  62  63  64  65  66  67  68  69  71  72  73  74  75  77  78  79  80
  82  83  84  85  87  88  89  91  92  93  95  96  98  99 100 102 103 105
 106 108 109 111 112 114 115 117 119 120 122 123 125 127 128 130 132 133
 135 137 138 140 142 144 145 147 149 151 153 155 156 158 160 162 164 166
 168 170 172 174 176 178 180 182 184 186 188 190 192 194 197 199 201 203
 205 207 210 212 214 216 219 221 223 226 228 230 23